# Modul ADV-07: Vision Foundation Models & Parameter-Efficient Adaptation
## Hands-on Laboratory: Contrastive InfoNCE Loss, Cosine Alignment, and LoRA Fine-Tuning

**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Level:** L3 Advanced Curriculum / Research Track R04 Preparation

Laboratorium komputasi ini mengimplementasikan konsep utama model fondasi visi:
1. **Penyelarasan Multimodal (CLIP-style):** Metrik kesamaan sudut Cosine dan normalisasi $L_2$.
2. **InfoNCE Contrastive Loss:** Estimasi batas bawah informasi mutual dan dampak penskalaan temperatur $\tau$.
3. **Zero-Shot Transfer:** Probabilitas klasifikasi berbasis Softmax berskala temperatur.
4. **Low-Rank Adaptation (LoRA):** Dekomposisi bobot adaptasi $\Delta W = \frac{\alpha}{r} (B \cdot A)$ dan evaluasi penghematan parameter.

In [ ]:
import numpy as np

print("NumPy Version:", np.__version__)
print("Environment initialized successfully for ADV-07 Foundation Models Lab.")

In [ ]:
def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norm, eps)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(l2_normalize(a), l2_normalize(b)))

# Uji coba vektor ortogonal, identik, dan berlawanan
v_ident = cosine_similarity(np.array([1, 0, 0]), np.array([1, 0, 0]))
v_ortho = cosine_similarity(np.array([1, 0, 0]), np.array([0, 1, 0]))
v_oppos = cosine_similarity(np.array([1, 0, 0]), np.array([-1, 0, 0]))

print(f"Cosine Identik:   {v_ident:.4f}")
print(f"Cosine Ortogonal: {v_ortho:.4f}")
print(f"Cosine Berlawanan: {v_oppos:.4f}")

In [ ]:
def compute_infonce_loss(query, pos_key, neg_keys, tau=0.07):
    q_norm = l2_normalize(query)
    pos_norm = l2_normalize(pos_key)
    neg_norm = l2_normalize(neg_keys)
    
    pos_sim = np.dot(q_norm, pos_norm)
    pos_logit = pos_sim / tau
    neg_logits = np.dot(neg_norm, q_norm) / tau
    
    all_logits = np.concatenate([[pos_logit], neg_logits])
    max_logit = np.max(all_logits)
    sum_exp = np.sum(np.exp(all_logits - max_logit))
    loss = (max_logit + np.log(sum_exp)) - pos_logit
    pos_prob = np.exp(pos_logit - max_logit) / sum_exp
    return float(loss), float(pos_sim), float(pos_prob)

q = np.array([0.82, 0.45, -0.12, 0.05])
pos_k = np.array([0.80, 0.48, -0.09, 0.02])
negs = np.array([
    [-0.15, 0.78, 0.45, -0.21],
    [-0.35, -0.12, 0.82, 0.41],
    [0.10, -0.22, 0.35, -0.75]
])

for tau_val in [0.01, 0.07, 0.20, 1.00]:
    l, sim, p = compute_infonce_loss(q, pos_k, negs, tau=tau_val)
    print(f"Temperatur tau={tau_val:.2f} | Loss: {l:.4f} | Probabilitas Positif: {p*100:.2f}%")

In [ ]:
# Zero-Shot Classification via Cosine Retrieval
text_prompts = [
    "a photo of a tiger in Sumatran jungle",
    "an oil palm plantation in Lampung",
    "a ship navigating Sunda strait"
]
candidate_embeddings = np.array([
    [0.80, 0.48, -0.09, 0.02], # Kelas 0 (Cocok)
    [-0.15, 0.78, 0.45, -0.21], # Kelas 1
    [-0.35, -0.12, 0.82, 0.41]  # Kelas 2
])

sims = np.array([cosine_similarity(q, c) for c in candidate_embeddings])
tau = 0.1
exp_sims = np.exp(sims / tau)
probs = exp_sims / np.sum(exp_sims)

print("Hasil Prediksi Zero-Shot:")
for i, (prompt, prob) in enumerate(zip(text_prompts, probs)):
    print(f"  Prompt [{i}]: '{prompt}' -> Probabilitas: {prob*100:.2f}%")

In [ ]:
# Low-Rank Adaptation (LoRA) Weight Matrix Decomposition
W0 = np.array([
    [1.0, 0.0, 2.0, 0.0],
    [0.0, 1.0, 0.0, 2.0],
    [2.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 1.0]
], dtype=np.float64)

# Inisialisasi LoRA: B diinisialisasi 0, A diinisialisasi acak
r = 1
alpha = 2.0
B = np.array([[0.5], [1.0], [-0.5], [0.0]], dtype=np.float64)
A = np.array([[2.0, 1.0, 0.0, -1.0]], dtype=np.float64)

delta_W = (alpha / r) * (B @ A)
W_adapted = W0 + delta_W

print("Bobot Awal W0:\n", W0)
print("Delta W (LoRA):\n", delta_W)
print("Bobot Teradaptasi W:\n", W_adapted)

In [ ]:
# Analisis Penghematan Parameter pada Arsitektur Vision Transformer Populer
models = [
    ("ViT-Base (d=768)", 768, 768, 4),
    ("ViT-Large (d=1024)", 1024, 1024, 8),
    ("ViT-Huge (d=1280)", 1280, 1280, 8)
]

print("Analisis Efisiensi Parameter LoRA pada Lapisan Proyeksi Atensi:")
for name, din, dout, rank in models:
    full_p = din * dout
    lora_p = (din + dout) * rank
    pct = ((full_p - lora_p) / full_p) * 100
    print(f"  {name} | Full Params: {full_p:,} | LoRA Params: {lora_p:,} | Penghematan: {pct:.2f}%")

### Kesimpulan Laboratorium & Keterkaitan Riset (Track R04)
1. **InfoNCE Loss** memaksimalkan informasi mutual antara ruang modalitas visual dan bahasa melalui dot product berskala temperatur.
2. **Temperatur $\tau$** bertindak sebagai parameter regulasi kekerasan distribusi; $\tau \approx 0.07$ menghasilkan penyejajaran optimal.
3. **LoRA** mereduksi parameter yang harus disimpan dan diperbarui sebesar $>98\%$, mencegah *catastrophic forgetting* pada data lokal Sumatera.
4. Formulasi ini menjadi landasan metodologis bagi **Research Track R04** (*Vision-Centric Multimodal Perception*).